# promo-finder in Colab

Finds the 20-60 s promo clips in a micro-drama's timestamped scripts: exact cut points, the hook, the cliffhanger, why it works, which episodes to take them from, the question each promo plants and where the show answers it.

Run the cells top to bottom. Without a key it runs the keyword twin (instant); with a free OpenRouter key it uses NVIDIA Nemotron 3.5 Super, an open-weights model, at no cost. Code: https://github.com/Raakshass/promo-finder

In [ ]:
!git clone -q --depth 1 https://github.com/Raakshass/promo-finder
%cd promo-finder
!pip install -q -e .

In [ ]:
# 1. The show: upload a .zip of timestamped screenplays, or press Cancel to use the bundled 2-episode synthetic demo
import pathlib, zipfile
SHOW = 'tests/fixtures'
try:
    from google.colab import files
    up = files.upload()
    if up:
        name = next(iter(up))
        pathlib.Path('show').mkdir(exist_ok=True)
        if name.endswith('.zip'):
            zipfile.ZipFile(name).extractall('show')
        else:
            pathlib.Path('show', name).write_bytes(up[name])
        SHOW = 'show'
except Exception as e:
    print('no upload, using the synthetic demo:', e)
!promo-finder parse {SHOW}

In [ ]:
# 2. The model. Free: an OpenRouter key (openrouter.ai/keys). Any provider works: gemini/..., groq/..., openai/...,
#    anthropic/..., or custom/<model> with PROMO_BASE_URL for an OpenAI-compatible endpoint. Leave the key empty for no model.
import os, getpass
MODEL = 'openrouter/nvidia/nemotron-3.5-super-vl:free'
if not os.environ.get('OPENROUTER_API_KEY'):
    os.environ['OPENROUTER_API_KEY'] = getpass.getpass('OpenRouter key (Enter = run without a model): ')
if not os.environ.get('OPENROUTER_API_KEY'):
    MODEL = 'none'
print('model:', MODEL)

In [ ]:
# 3. Run: free tiers cap requests per minute, so batch 5 episodes per call
!promo-finder run {SHOW} --model {MODEL} --batch 5 --reasoning-off --out /content/outputs

In [ ]:
# 4. The ranked promos, the promise & payoff ledger and the promo-ability report
import glob
from IPython.display import Markdown, display
run = sorted(glob.glob('/content/outputs/*/*'))[-1]
display(Markdown(open(f'{run}/promos.md').read()[:30000]))
display(Markdown(open(f'{run}/eval/promoability.md').read()[:8000]))
print('All files:', run)